In [65]:
import pandas as pd

# Trabalho 1 - Rafael Gabriel 121087555

## Introdução

O trabalho 1 consiste da criação de um classificador para apoio na decisão de aprovação de crédito. Esse trabalho está separado em dois grupos:
- Análise exploratória dos dados;
- Tratamento dos dados;
- Treinamento e validação;
- Predição e envio. 

## Análise exploratória dos dados

Essa etapa consiste da análise dos dados, verificação se há outliers, colunas com valores em branco e etc. O primeiro passo dessa etapa é a importação dos dados para um dataframe do pandas. A coluna ```id_solicitante``` já foi removida pois não traz nenhuma informação.

In [66]:
df = pd.read_csv("data/conjunto_de_treinamento.csv",  na_values=[' ', 'NULL']).drop(columns='id_solicitante')
df.describe().T

,count,mean,std,min,25%,50%,75%,max
produto_solicitado,20000.0,1.284350,1.008239,1.0,1.0,1.0,1.0,7.0
dia_vencimento,20000.0,13.147250,6.748507,1.0,10.0,10.0,20.0,25.0
tipo_endereco,20000.0,1.006350,0.079435,1.0,1.0,1.0,1.0,2.0
idade,20000.0,42.352500,14.930177,7.0,31.0,40.0,52.0,106.0
estado_civil,20000.0,2.120850,1.332004,0.0,1.0,2.0,2.0,7.0
qtde_dependentes,20000.0,0.666400,1.236725,0.0,0.0,0.0,1.0,53.0
grau_instrucao,20000.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
nacionalidade,20000.0,0.961600,0.202305,0.0,1.0,1.0,1.0,2.0
codigo_area_telefone_residencial,16466.0,64.362626,38.701585,1.0,25.0,69.0,100.0,126.0
tipo_residencia,19464.0,1.261303,0.883580,0.0,1.0,1.0,1.0,5.0


A partir do dicionário de dados (dicionario_de_dados.xlsx) é possível identificar que a coluna grau_instrucao possui apenas valores 0. Por precaução, foi procurado mais colunas com possíveis valores iguais.

In [67]:
valoresDistintos = df.nunique(dropna=False).sort_values()

tabelaDistintos = pd.DataFrame({
    "Valores distintos": valoresDistintos,
    "Exemplo de valor": df[valoresDistintos.index].iloc[0],
})

# Drop das colunas que possuem apenas 1 valor distinto
colunasConstantes = tabelaDistintos[tabelaDistintos["Valores distintos"] == 1].index.tolist()
df = df.drop(columns=colunasConstantes)

tabelaDistintos

,Valores distintos,Exemplo de valor
grau_instrucao,1,0
possui_telefone_celular,1,N
possui_telefone_residencial,2,Y
tipo_endereco,2,1
possui_cartao_visa,2,0
possui_cartao_mastercard,2,0
possui_carro,2,1
possui_cartao_diners,2,0
possui_outros_cartoes,2,0
possui_telefone_trabalho,2,N


Pelo resultado, é possível notar que além da coluna ```grau_instrucao``` a coluna ```possui_telefone_celular``` também possui todos os valores iguais a N, por isso foi removida também.

A partir do método ```describe``` também é possível notar que a coluna ```local_onde_reside``` é idêntica a coluna ```local_onde_trabalha```. Além de remover uma delas, já que duas colunas iguais não acrescentam nenhuma informação extra, também foi realizado a procura por mais colunas iguais.

In [68]:
colunas = df.columns.tolist()
colunasIguais = []

for i in range(len(colunas)):
    for j in range(i + 1, len(colunas)):
        if df[colunas[i]].equals(df[colunas[j]]):
            colunasIguais.append((colunas[i], colunas[j]))


# Drop primeira coluna dos pares de colunas iguais
colunasDuplicada = [primeiraColunas[0] for primeiraColunas in colunasIguais]
df = df.drop(columns=colunasDuplicada)
colunasIguais



[('qtde_contas_bancarias', 'qtde_contas_bancarias_especiais'),
 ('local_onde_reside', 'local_onde_trabalha')]

A partir do código é possível notar que existe mais um par de colunas iguais: o par ```('qtde_contas_bancarias', 'qtde_contas_bancarias_especiais')```. Assim como para o local onde reside/trabalha, uma das colunas foi removida. 

Uma coluna que foi decidida tirar também foi a coluna de ```meses_no_trabalho```, pois, conforme a descirção no dicionário, a grande maioria está com 0, mesmo com vínculo com uma empresa. Com isso, ficamos com o seguinte DF

In [69]:
df = df.drop(columns='meses_no_trabalho')
df.describe().T

,count,mean,std,min,25%,50%,75%,max
produto_solicitado,20000.0,1.284350,1.008239,1.0,1.0,1.0,1.0,7.0
dia_vencimento,20000.0,13.147250,6.748507,1.0,10.0,10.0,20.0,25.0
tipo_endereco,20000.0,1.006350,0.079435,1.0,1.0,1.0,1.0,2.0
idade,20000.0,42.352500,14.930177,7.0,31.0,40.0,52.0,106.0
estado_civil,20000.0,2.120850,1.332004,0.0,1.0,2.0,2.0,7.0
qtde_dependentes,20000.0,0.666400,1.236725,0.0,0.0,0.0,1.0,53.0
nacionalidade,20000.0,0.961600,0.202305,0.0,1.0,1.0,1.0,2.0
codigo_area_telefone_residencial,16466.0,64.362626,38.701585,1.0,25.0,69.0,100.0,126.0
tipo_residencia,19464.0,1.261303,0.883580,0.0,1.0,1.0,1.0,5.0
meses_na_residencia,18550.0,9.572453,10.649580,0.0,1.0,6.0,15.0,228.0


Partindo para a análise por linhas, o primeiro passo foi identificar os *outliers*. É possível identificar eles usando o percentil.

In [70]:
df.describe(percentiles=[0.01, 0.5, 0.99]).T

,count,mean,std,min,1%,50%,99%,max
produto_solicitado,20000.0,1.284350,1.008239,1.0,1.0,1.0,7.00,7.0
dia_vencimento,20000.0,13.147250,6.748507,1.0,1.0,10.0,25.00,25.0
tipo_endereco,20000.0,1.006350,0.079435,1.0,1.0,1.0,1.00,2.0
idade,20000.0,42.352500,14.930177,7.0,18.0,40.0,80.00,106.0
estado_civil,20000.0,2.120850,1.332004,0.0,1.0,2.0,7.00,7.0
qtde_dependentes,20000.0,0.666400,1.236725,0.0,0.0,0.0,5.00,53.0
nacionalidade,20000.0,0.961600,0.202305,0.0,0.0,1.0,1.00,2.0
codigo_area_telefone_residencial,16466.0,64.362626,38.701585,1.0,5.0,69.0,125.00,126.0
tipo_residencia,19464.0,1.261303,0.883580,0.0,0.0,1.0,5.00,5.0
meses_na_residencia,18550.0,9.572453,10.649580,0.0,0.0,6.0,45.00,228.0


Com a tabela acima é possível identificar algumas colunas com possíveis *outliers*, sendo elas:
- ```idade```: solicitante com 106 anos;
- ```qtde_dependentes```: solicitante com 53 dependentes;
- ```meses_na_residencia```: solicitante com 228 meses na residência;
- ```renda_mensal_regular```: solicitante com renda mensal de 959000 reais;
- ```renda_extra```: solicitante com renda extra de 194344 reais;
- ```valor_patrimonio_pessoal```: solicitante com patrimonio de 6000000 reais;

# ANALISE DE NANS, MUITO CHATO DE DEPENDE DO MODELO QUE VAMOS ESCOLHER

Partindo para a análise por linhas, o primeiro passo foi procurar linhas com valores NaN para decidir o que seria feito com elas. Importante pontuar que  não tem linhas em branco por conta do argumento que usei para importar o csv para o dataframe.

In [71]:
tabelaAusentes = pd.DataFrame({
    "Valores preenchidos": df.notna().sum(),
    "NaN": df.isna().sum(),
    "% NaN": (df.isna().mean() * 100).round(1),
})


tabelaAusentes = tabelaAusentes[tabelaAusentes["NaN"] > 0].sort_values("NaN", ascending=False)
tabelaAusentes

,Valores preenchidos,NaN,% NaN
codigo_area_telefone_trabalho,5475,14525,72.6
estado_onde_trabalha,6427,13573,67.9
grau_instrucao_companheiro,7140,12860,64.3
profissao_companheiro,8486,11514,57.6
codigo_area_telefone_residencial,16466,3534,17.7
profissao,16903,3097,15.5
ocupacao,17022,2978,14.9
meses_na_residencia,18550,1450,7.2
estado_onde_nasceu,19178,822,4.1
tipo_residencia,19464,536,2.7


A primeira análise é relacionado as colunas ```codigo_area_telefone_trabalho``` e ```estado_onde_trabalha```, pois podem com NaN pela fato da pessoa não ter emprego. Para fazer essa análise foi utilizado a coluna ```vinculo_formal_com_empresa```.

In [72]:
pd.crosstab(df["vinculo_formal_com_empresa"], df["codigo_area_telefone_trabalho"].isna())

codigo_area_telefone_trabalho,False,True
vinculo_formal_com_empresa,,
N,397,10777
Y,5078,3748


In [73]:
pd.crosstab(df["vinculo_formal_com_empresa"], df["estado_onde_trabalha"].isna())

estado_onde_trabalha,False,True
vinculo_formal_com_empresa,,
N,346,10828
Y,6081,2745


# VER O QUE VAI FAZER AQUI 